## 0. Environment Setup

#### Import libraries

In [0]:
'Import libraries'
import json, time, pytz, requests
from datetime import datetime, timedelta
import pandas as pd

## 1. Data Retrieval

In [0]:
base_url = "https://api.openelectricity.org.au/v4"
api_key = dbutils.secrets.get(scope="nem_project", key="openelectricity_api_key") #secret API key

### 1.1 Get All Facility Codes
Get the list of facilities

In [0]:
'Facility Information retrieval'
#forge the url
url_facility = f"{base_url}/facilities/"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Accept": "application/json"
}

#Filter for NEM network (Operating in New South Wales, the Australian Capital Territory, Queensland, South Australia, Victoria and Tasmania)
params = {"network_id": ["NEM"]}

response_facility = requests.get(url_facility, headers=headers, params=params)

print("Status code:", response_facility.status_code)

#Print the retrieved facilities
if response_facility.status_code == 200:
    facilities = response_facility.json()
    print(json.dumps(facilities, indent=2))
else:
    print("Error:", response_facility.text)

facilities = response_facility.json()

In [0]:
'Save facilities list to file'

#Persist to disk so downstream sections don't depend on the in-memory 'facilities' variable
with open("facilities.json", "w") as f:
    json.dump(facilities, f, indent=2)
print("Saved to facilities.json")

In [0]:
'Number of facilities retrieved'
num_facilities = len(facilities["data"])
print("Total facilities:", num_facilities)

## 1.2 Query for power generated and CO2 emissions per facility (1H interval)

In [0]:
'Power and Emissions Data Retrieval by Facility'

#Set up API parameters
network_code = "NEM"
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {api_key}", "Accept": "application/json"})

#Extract all facility codes
facility_codes = [f["code"] for f in facilities["data"]]

#Query power & emissions data for 01/09/2025 - 31/08/2026, hourly interval
tz = pytz.timezone("Australia/Sydney")
date_start = tz.localize(datetime(2025, 9, 1, 0, 0, 0)).replace(tzinfo=None)
date_end = tz.localize(datetime(2026, 8, 31, 23, 59, 59)).replace(tzinfo=None)
interval = "1h"

#use facilities API
data_url = f"{base_url}/data/facilities/{network_code}"
output_path = "facility_power_emissions_sep2025_aug2026.json"

#The API caps a 1h-interval request to a 32-day window, so split the year into <=32-day chunks
#(same approach as section 1.3). With 553 facilities x 12 chunks this cell issues ~6,600
#requests, so expect it to take a while to finish.
MAX_CHUNK_DAYS = 32
def date_chunks(start, end, max_days=MAX_CHUNK_DAYS):
    span = timedelta(days=max_days) - timedelta(seconds=1)
    chunks = []
    chunk_start = start
    while chunk_start <= end:
        chunk_end = min(chunk_start + span, end)
        chunks.append((chunk_start, chunk_end))
        chunk_start = chunk_end + timedelta(seconds=1)
    return chunks

windows = date_chunks(date_start, date_end)
print(f"Split {date_start.date()} - {date_end.date()} into {len(windows)} chunk(s) of <= {MAX_CHUNK_DAYS} days")

#Merge same-metric chunk responses for one facility into a single payload,
#concatenating each result's time series and widening the reported date range
def merge_facility_chunks(chunk_responses):
    metric_blocks = {}
    for resp in chunk_responses:
        for block in resp.get("data", []):
            metric = block["metric"]
            if metric not in metric_blocks:
                metric_blocks[metric] = {
                    "network_code": block["network_code"],
                    "metric": metric,
                    "unit": block["unit"],
                    "interval": block["interval"],
                    "date_start": block["date_start"],
                    "date_end": block["date_end"],
                    "groupings": block.get("groupings", []),
                    "network_timezone_offset": block.get("network_timezone_offset"),
                    "results_by_name": {},
                }
            merged_block = metric_blocks[metric]
            merged_block["date_start"] = min(merged_block["date_start"], block["date_start"])
            merged_block["date_end"] = max(merged_block["date_end"], block["date_end"])

            for r in block.get("results", []):
                name = r["name"]
                if name not in merged_block["results_by_name"]:
                    merged_block["results_by_name"][name] = {
                        "name": name,
                        "date_start": r["date_start"],
                        "date_end": r["date_end"],
                        "columns": r["columns"],
                        "data": list(r["data"]),
                    }
                else:
                    merged_result = merged_block["results_by_name"][name]
                    merged_result["data"].extend(r["data"])
                    merged_result["date_end"] = max(merged_result["date_end"], r["date_end"])

    merged_data = []
    for block in metric_blocks.values():
        block["results"] = list(block.pop("results_by_name").values())
        merged_data.append(block)

    return {"success": True, "data": merged_data}

MAX_RETRIES = 3
RETRY_BACKOFF = 2.0  #seconds, doubled on each retry
REQUEST_PAUSE = 0.2  #pause between successful requests to avoid overwhelming the API

results = {}
failed_codes = []

#Loop through each facility, then through each date chunk, retrying transient failures
for i, code in enumerate(facility_codes, 1):
    print(f"[{i}/{num_facilities}] {code}: ", end="")
    chunk_responses = []
    facility_failed = False

    for w, (chunk_start, chunk_end) in enumerate(windows, 1):
        params = {
            "metrics": ["power", "emissions"],
            "interval": interval,
            "facility_code": [code],
            "date_start": chunk_start.isoformat(),
            "date_end": chunk_end.isoformat(),
        }

        response_json = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = session.get(data_url, params=params, timeout=30)
            except requests.RequestException as exc:
                print(f"\n  chunk {w}/{len(windows)}: request error ({exc}), retry {attempt}/{MAX_RETRIES}", end="")
                time.sleep(RETRY_BACKOFF * attempt)
                continue

            #Retry on rate limiting or server-side errors
            if response.status_code in (429, 500, 502, 503, 504):
                print(f"\n  chunk {w}/{len(windows)}: {response.status_code}, retry {attempt}/{MAX_RETRIES}", end="")
                time.sleep(RETRY_BACKOFF * attempt)
                continue

            try:
                response_json = response.json()
            except json.JSONDecodeError:
                print(f"\n  chunk {w}/{len(windows)}: error decoding JSON", end="")
                response_json = None
            break

        if response_json is None:
            print(f"\n  chunk {w}/{len(windows)}: giving up after {MAX_RETRIES} attempts", end="")
            facility_failed = True
            time.sleep(REQUEST_PAUSE)
            continue

        #Check if the API returned success
        if response.status_code == 200 and response_json.get("success"):
            if response_json.get("data"):
                chunk_responses.append(response_json)
            #a chunk with no data is fine (e.g. facility not yet commissioned in that window)
        else:
            print(f"\n  chunk {w}/{len(windows)}: error: {response_json.get('error', 'Unknown error')}", end="")
            facility_failed = True

        #Pause to avoid overwhelming the API
        time.sleep(REQUEST_PAUSE)

    if chunk_responses:
        results[code] = merge_facility_chunks(chunk_responses)
        print(f"✅ {len(chunk_responses)}/{len(windows)} chunks merged")
    else:
        print("⚠️ no data returned across all chunks")

    if facility_failed:
        failed_codes.append(code)

print(f"\nCompleted: {len(results)}/{num_facilities} facilities retrieved, {len(failed_codes)} failed/partial")
if failed_codes:
    print("Failed/partial facility codes:", failed_codes)

#save all results to file
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved to {output_path}")

In [0]:
#check each response
response_json

In [0]:
#check the final concatenation
results

## 1.3 Query for per-market price and demand (1H interval)

The API only gives us 32 days of hourly data per request, but we want a full year. So for each region we break the year into 32-day pieces, request each piece one by one, then merge them back together into a single file per region.

In [0]:
'Price and Demand Data Retrieval by Region'

#Reuse a session (consistent with the facility retrieval cell, and lets us reuse the TCP connection across requests)
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {api_key}", "Accept": "application/json"})

network_code = "NEM"

#Derive NEM regions from 'facilities' (supports either a DataFrame or the raw API dict)
def get_network_regions(facilities_obj, network_code: str):
    #Case 1: facilities is a pandas DataFrame
    if isinstance(facilities_obj, pd.DataFrame):
        regions = (
            facilities_obj.loc[
                facilities_obj["network_id"].astype(str).eq(network_code), "network_region"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )
        return sorted(regions)

    #Case 2: facilities is a dict with a 'data' list, as returned by the API
    if isinstance(facilities_obj, dict) and "data" in facilities_obj:
        regions = {
            f.get("network_region")
            for f in facilities_obj["data"]
            if f.get("network_id") == network_code and f.get("network_region")
        }
        return sorted(regions)

    #Fallback if neither shape matched
    return []

network_regions = get_network_regions(facilities, network_code)
if not network_regions:
    #Safe default for NEM
    network_regions = ["NSW1", "QLD1", "SA1", "VIC1", "TAS1"]

print("Network regions:", network_regions)

#Query price & demand for 01/09/2025 - 31/08/2026, hourly interval (same window as section 1.2)
tz = pytz.timezone("Australia/Sydney")
date_start = tz.localize(datetime(2025, 9, 1, 0, 0, 0)).replace(tzinfo=None)
date_end   = tz.localize(datetime(2026, 8, 31, 23, 59, 59)).replace(tzinfo=None)
interval = "1h"

#Endpoint for network-level metrics
data_url = f"{base_url}/market/network/{network_code}"
output_path = "region_price_demand_sep2025_aug2026.json"

#Simple explanation: the API only gives us 32 days of hourly data per request,
#but we want a full year. So for each region we break the year into 32-day
#pieces, request each piece one by one, then glue (merge) them back together
#into a single file per region.
#The API caps a 1h-interval request to a 32-day window, so split the year into <=32-day chunks
MAX_CHUNK_DAYS = 32
def date_chunks(start, end, max_days=MAX_CHUNK_DAYS):
    span = timedelta(days=max_days) - timedelta(seconds=1)
    chunks = []
    chunk_start = start
    while chunk_start <= end:
        chunk_end = min(chunk_start + span, end)
        chunks.append((chunk_start, chunk_end))
        chunk_start = chunk_end + timedelta(seconds=1)
    return chunks

windows = date_chunks(date_start, date_end)
print(f"Split {date_start.date()} - {date_end.date()} into {len(windows)} chunk(s) of <= {MAX_CHUNK_DAYS} days")

#Merge same-metric chunk responses for one region into a single payload,
#concatenating each result's time series and widening the reported date range
def merge_region_chunks(chunk_responses):
    metric_blocks = {}
    for resp in chunk_responses:
        for block in resp.get("data", []):
            metric = block["metric"]
            if metric not in metric_blocks:
                metric_blocks[metric] = {
                    "network_code": block["network_code"],
                    "metric": metric,
                    "unit": block["unit"],
                    "interval": block["interval"],
                    "date_start": block["date_start"],
                    "date_end": block["date_end"],
                    "groupings": block.get("groupings", []),
                    "network_timezone_offset": block.get("network_timezone_offset"),
                    "results_by_name": {},
                }
            merged_block = metric_blocks[metric]
            merged_block["date_start"] = min(merged_block["date_start"], block["date_start"])
            merged_block["date_end"] = max(merged_block["date_end"], block["date_end"])

            for r in block.get("results", []):
                name = r["name"]
                if name not in merged_block["results_by_name"]:
                    merged_block["results_by_name"][name] = {
                        "name": name,
                        "date_start": r["date_start"],
                        "date_end": r["date_end"],
                        "columns": r["columns"],
                        "data": list(r["data"]),
                    }
                else:
                    merged_result = merged_block["results_by_name"][name]
                    merged_result["data"].extend(r["data"])
                    merged_result["date_end"] = max(merged_result["date_end"], r["date_end"])

    merged_data = []
    for block in metric_blocks.values():
        block["results"] = list(block.pop("results_by_name").values())
        merged_data.append(block)

    return {"success": True, "data": merged_data}

MAX_RETRIES = 3
RETRY_BACKOFF = 2.0   #seconds, doubled on each retry
REQUEST_PAUSE = 0.5   #pause between successful requests to avoid overwhelming the API

results_by_region = {}
failed_regions = []
n = len(network_regions)

#Loop through each region, then through each date chunk, retrying transient failures
for i, region in enumerate(network_regions, 1):
    print(f"[{i}/{n}] {region}:")
    chunk_responses = []
    region_failed = False

    for w, (chunk_start, chunk_end) in enumerate(windows, 1):
        params = {
            "metrics": ["price", "demand"],
            "interval": interval,
            "network_region": region,
            "date_start": chunk_start.isoformat(),
            "date_end": chunk_end.isoformat(),
        }

        print(f"  chunk {w}/{len(windows)} ({chunk_start.date()} - {chunk_end.date()}): ", end="")

        response_json = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = session.get(data_url, params=params, timeout=30)
            except requests.RequestException as exc:
                print(f"⚠️ request error ({exc}), retry {attempt}/{MAX_RETRIES}", end=" ")
                time.sleep(RETRY_BACKOFF * attempt)
                continue

            #Retry on rate limiting or server-side errors
            if response.status_code in (429, 500, 502, 503, 504):
                print(f"⚠️ {response.status_code}, retry {attempt}/{MAX_RETRIES}", end=" ")
                time.sleep(RETRY_BACKOFF * attempt)
                continue

            try:
                response_json = response.json()
            except json.JSONDecodeError:
                print(f"❌ error decoding JSON for {region}")
                response_json = None
            break

        if response_json is None:
            print(f"❌ giving up on this chunk after {MAX_RETRIES} attempts")
            region_failed = True
            time.sleep(REQUEST_PAUSE)
            continue

        #Validate success and presence of data
        if response.status_code == 200 and response_json.get("success"):
            if response_json.get("data"):
                chunk_responses.append(response_json)
                print(f"✅ {len(response_json['data'])} items")
            else:
                print("⚠️ no data returned")
        else:
            print(f"❌ error: {response_json.get('error', 'Unknown error')}")
            region_failed = True

        #Pause to avoid overwhelming the API
        time.sleep(REQUEST_PAUSE)

    if chunk_responses:
        results_by_region[region] = merge_region_chunks(chunk_responses)
    if region_failed or not chunk_responses:
        failed_regions.append(region)

print(f"\nCompleted: {len(results_by_region)}/{n} regions retrieved, {len(failed_regions)} failed")
if failed_regions:
    print("Failed/partial regions:", failed_regions)

#Save results per region
with open(output_path, "w") as f:
    json.dump(results_by_region, f, indent=2)
print(f"Saved to {output_path}")

In [0]:
#check each response
results_by_region

## Output files
The raw_data catalog produces three files:
* **facilities.json**: the station list and their metadata (region, fuel type, capacity, location, etc.)
* **facility_power_emissions_sep2025_aug2026.json**: power and emissions per station, hourly, from 01/09/2025 to 31/08/2026.
* **region_price_demand_sep2025_aug2026.json**: price and demand per NEM region, hourly, from 01/09/2025 to 31/08/2026.
